# Claims Triage Validation

Install requirements

In [1]:
%pip install -r requirements.txt --no-cache-dir --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.7/993.7 kB 361.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 394.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 465.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 649.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 526.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 174.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 496.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 350.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 561.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 355.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 144.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
# Import libraries
import os
import json
import requests
import boto3
import time
from boto3.session import Session
from strands.tools import tool

# Get boto session
boto_session = Session()

## Setup for the agent

### Create Code for the Agent
Create agents folder if it's not created.

In [2]:
%rm -rf ./agents >> /dev/null
%mkdir ./agents
%rm Dockerfile >> /dev/null

In [3]:
# Capture the environmental varaibles that were passed in from the lifecycle
BEDROCK_REGION = os.environ.get('BEDROCK_REGION')
ARC_POLICY_ARN = os.environ.get('ARC_POLICY_ARN')
GUARDRAIL_ID = os.environ.get('GUARDRAIL_ID')
GUARDRAIL_VERSION = os.environ.get('GUARDRAIL_VERSION')
KNOWLEDGE_BASE_ID = os.environ.get('KNOWLEDGE_BASE_ID')

In [4]:

ARC_POLICY_ARN

'arn:aws:bedrock:us-east-1:161615149547:automated-reasoning-policy/6if243lxvpvp:1'

In [5]:
GUARDRAIL_ID

'tqiexl44264z'

In [6]:
GUARDRAIL_VERSION

'DRAFT'

In [7]:
%%writefile agents/triage-agent.py
import os
import logging
import asyncio
import boto3
import json
import uvicorn
from datetime import datetime
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from strands.multiagent.a2a import A2AServer
from fastapi import FastAPI
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, BeforeModelCallEvent, BeforeToolCallEvent
from pydantic import BaseModel
from botocore.config import Config as BotocoreConfig
from strands.telemetry import StrandsTelemetry
from findings_utils import extract_reasoning_findings
from strands_tools import retrieve

# Configure the root strands logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

# Setup tracing - commented out for now as this adds a lot of trace output that really isn't interesting
# NOTE: To send the OTEL data to an ADOT collector, additional exporter needs to be used
StrandsTelemetry().setup_console_exporter()

# The default values as pulled fomr the environment.  These values
# are replaced after the file is written out.
BEDROCK_REGION = '<REPLACE_BEDROCK_REGION>'
ARC_POLICY_ARN = 'arn:aws:bedrock:us-east-1:161615149547:automated-reasoning-policy/k46bmidxs190'
GUARDRAIL_ID = '3fxp58y9y1dp'
GUARDRAIL_VERSION = '<REPLACE_GUARDRAIL_VERSION>'
KNOWLEDGE_BASE_ID = '<REPLACE_KNOWLEDGE_BASE_ID>'

# Initialize Bedrock client to verify connectivity
bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=BEDROCK_REGION
)

print(f"✓ AWS Bedrock configured for region: {BEDROCK_REGION}")

# Setup the environment for the agent and tool
# Allow for the metadata to be retrieved on sources from the KB
os.environ['RETRIEVE_ENABLE_METADATA_DEFAULT'] = 'true'

# Store the Knowledge Base ID in the environment to allow the tool
# to interact with the KB
os.environ['KNOWLEDGE_BASE_ID'] = KNOWLEDGE_BASE_ID

# Define a notification hook to listen to events and then process the result and call
# Automated Reasoning attached via the Guardrail and report on the findings.  This
# can be used possibly re-write the output or add a flag on if the output is correct.
# Define a notification hook to listen to events and then process the result and call
# Automated Reasoning attached via the Guardrail and report on the findings.  This
# can be used possibly re-write the output or add a flag on if the output is correct.
class NotifyOnlyGuardrailsHook(HookProvider):
    
    def __init__(self, guardrail_id: str, guardrail_version: str, arc_policy_arn: str):
        self.guardrail_id = guardrail_id
        self.guardrail_version = guardrail_version
        self.arc_policy_arn = arc_policy_arn
        self.bedrock_client = boto3.client("bedrock-runtime")
        self.input = ''
        self.claim_valid = True
        self.findings = ''
        self.policy_definition = {}
        self.before_tool_event_flag = False
        self.before_model_event_flag = False

        if self.arc_policy_arn:
            try:
                bedrock_client = boto3.client('bedrock')
                response = bedrock_client.export_automated_reasoning_policy_version(policyArn=self.arc_policy_arn)
                self.policy_definition = response.get('policyDefinition', {})
            except Exception as e:
                print(f"Error getting policy definition: {str(e)}")
                raise

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeModelCallEvent, self.before_model_event)
        registry.add_callback(BeforeToolCallEvent, self.before_tool_event)
        registry.add_callback(MessageAddedEvent, self.message_added)

    def message_added(self, event: MessageAddedEvent) -> None:
        if self.before_tool_event_flag:
            # Since a tool was called, just ignore this message addition
            self.before_tool_event_flag = False
            return
        
        # Get the content
        content = "".join(block.get("text", "") for block in event.message.get("content", []))

        # Determine the source
        if event.message.get("role") == "user":
            # Store the input for later usage and allow the loop to continue to process
            self.input = content
            return

        if not content:
            return
            #do something 

        # Capture if this is the first time that findings will be created
        first_findings = (not self.findings)

        # Format a request to send to the guardrail
        content_to_validate = [
            {"text": {"text": self.input, "qualifiers": ["query"]}},
            {"text": {"text": content, "qualifiers": ["guard_content"]}}
        ]
        
        # Call the guardrail
        response = self.bedrock_client.apply_guardrail(
            guardrailIdentifier=self.guardrail_id,
            guardrailVersion=self.guardrail_version,
            source="OUTPUT",
            content=content_to_validate
        )

        # Determine if the output is correct
        self.findings = extract_reasoning_findings(response, self.policy_definition)
         
        assessments = response.get("assessments", [])
        if assessments and len(assessments):
            self.claim_valid = False

        # Add information to the output
        if self.findings and first_findings:
            new_output = content
            new_output = new_output + f"\n*** FINDINGS: ***:\n{self.findings}"
            new_output = new_output + f"\n*** CLAIM VALID: ***:\n{self.claim_valid}"
            event.message["content"][0]["text"] = new_output
        
    def before_model_event(self, event: BeforeModelCallEvent) -> None:
        self.before_model_event_flag = True

    def before_tool_event(self, event: BeforeToolCallEvent) -> None:
        self.before_tool_event_flag = True

# Create structured output
class StructuredOutputModel(BaseModel):
    claim_valid: bool
    content: str
    findings: str

agent_instructions="""You are an expert automotive claims validation specialist that determines if the users auto insurance claim is valid based on the provided information and details within the policy contract.
    
You will be provided with a triage report that has claim information and vehicle damage information, you should:
1. Extract from the triage report the required claim information to be validated
2. Focus on discrepancies with the claims and coverage inconsistencies
3. Focus on each field and vality of the field
4. Review the "Adjuster Question" and provide an answer 

Your responses should :
- If your response is "Valid claim", then output the provide the a report explain why is correct
- If you response is "Invalid claim", then output the response "This Triage Rerport is Invalid and needs to be escalated to a very expensive Lawyer"
- If you response is "Invalid claim", then provide clear explanation on why is invalid in the output
- In cases where a clear outcome is not present, recommend the user to check with their insurance agent directly. 
- "Adjuster Question" answer validation based on legal opinion using Automated Reasoning.

Create a section called "Triage Report Validation Explainability". Use the Automated Reasoning Checks rules to provide explanation and list the rules that were valid and not valid
Provide the following for this section:
he Automated Reasoning Checks rules to provide explanation and list the rules that were valid and not validz
## Finding 
**Finding Type:** 

### Translation:
#### Premises:
#### Claims:
#### Untranslated Claims:

**Confidence Score:** 

### Claims True Scenario:


Take your time to think though the answer and evalute carefully."

"""

# Create agent with the guardrail-protected model
agent = Agent(
    name="Legal Triage - ARC - NO RAG",
    description="A Single Agent with ARC Legal Triage Validation",
    hooks=[NotifyOnlyGuardrailsHook(GUARDRAIL_ID, GUARDRAIL_VERSION, ARC_POLICY_ARN)],
    # tools=[retrieve],
    tools=[],
    system_prompt=agent_instructions
)

################# A2A ################
app = FastAPI()
runtime_url = os.environ.get('AGENTCORE_RUNTIME_URL', 'http://127.0.0.1:9000/')
host, port = "0.0.0.0", 9000

a2a_server = A2AServer(
    agent=agent,
    http_url=runtime_url,
    serve_at_root=True,
)

# Respond to pings
@app.get("/ping")
def ping():
    return {"status": "healthy"}

# Any request, call the server
app.mount("/", a2a_server.to_fastapi_app())

# If this is running from main, start the server
if __name__ == "__main__":
    uvicorn.run(app, host=host, port=port)

################# A2A ################

Writing agents/triage-agent.py


In [8]:
# For the created file, replace the placeholder values with environmental data
def replace_text_in_file(file_path, old_string, new_string):
    # Read the file content
    with open(file_path, 'r') as file:
        file_contents = file.read()

    # Perform the replacement(s)
    # The string replace() method returns a *new* string
    updated_contents = file_contents.replace(old_string, new_string)

    # Write the modified content back to the same file
    with open(file_path, 'w') as file:
        file.write(updated_contents)

    print(f"Text replacement completed successfully in {file_path}.")

# Example usage:
# Assuming a file named 'example.txt' exists with content "This is an old house."
replace_text_in_file('agents/triage-agent.py', '<REPLACE_BEDROCK_REGION>', BEDROCK_REGION)
replace_text_in_file('agents/triage-agent.py', '<REPLACE_ARC_POLICY_ARN>', 'arn:aws:bedrock:us-east-1:161615149547:automated-reasoning-policy/k46bmidxs190')
replace_text_in_file('agents/triage-agent.py', '<REPLACE_GUARDRAIL_ID>', '3fxp58y9y1dp')
replace_text_in_file('agents/triage-agent.py', '<REPLACE_GUARDRAIL_VERSION>', GUARDRAIL_VERSION)
replace_text_in_file('agents/triage-agent.py', '<REPLACE_KNOWLEDGE_BASE_ID>', KNOWLEDGE_BASE_ID)

Text replacement completed successfully in agents/triage-agent.py.
Text replacement completed successfully in agents/triage-agent.py.
Text replacement completed successfully in agents/triage-agent.py.
Text replacement completed successfully in agents/triage-agent.py.
Text replacement completed successfully in agents/triage-agent.py.


In [9]:
# ARC_POLICY_ARN = 'arn:aws:bedrock:us-east-1:161615149547:automated-reasoning-policy/59wsgpic8hex'
# GUARDRAIL_ID = 'l48lfasgimti'

Let's write a requirements.txt file with dependencies that are needed for the agent.

In [10]:
%%writefile agents/requirements.txt
boto3
botocore
bedrock-agentcore
bedrock-agentcore-starter-toolkit
strands-agents
strands-agents[a2a]
strands-agents-tools
pyyaml
ddgs
PyYAML
PyPDF2
opentelemetry-sdk
opentelemetry-exporter-otlp
opentelemetry-instrumentation-boto
opentelemetry-sdk-extension-aws
pydantic 
httpx
fastapi
uvicorn[standard]
requests

Writing agents/requirements.txt


###  Deploy to AgentCore Runtime

#### Setup Cognito User Pool

In [11]:
from helpers.utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = (
    setup_cognito_user_pool()
)  # You'll get your bearer token from this output cell.
print("Cognito setup completed ✓")

Setting up Amazon Cognito user pool...
Pool id: us-east-1_JnnzjXKFq
Discovery URL: https://cognito-idp.us-east-1.amazonaws.com/us-east-1_JnnzjXKFq/.well-known/openid-configuration
Client ID: 1m2dej9j10v39aml3tcabdpl3t
Bearer Token: eyJraWQiOiJqeCtFeDhBZmxqSktpaGdGTFVWRGs5WGZpZ2dyNklVNjF5NU0yYjhsT3I4PSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiI3NDA4ODQ1OC00MDExLTcwMTYtMDk3Ny0zNDRiY2YxNTY4NGQiLCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAudXMtZWFzdC0xLmFtYXpvbmF3cy5jb21cL3VzLWVhc3QtMV9Kbm56alhLRnEiLCJjbGllbnRfaWQiOiIxbTJkZWo5ajEwdjM5YW1sM3RjYWJkcGwzdCIsIm9yaWdpbl9qdGkiOiI2MGVlZTNjMS1mMDY3LTQ4YjMtOTAyMy1iYjY5ODg3ZjhkNTkiLCJldmVudF9pZCI6ImI0MGE3MWJhLTU0MjgtNDJlZS05YzEyLWMzY2QwMjBkNmJhYiIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoiYXdzLmNvZ25pdG8uc2lnbmluLnVzZXIuYWRtaW4iLCJhdXRoX3RpbWUiOjE3NzUwNTE3NDQsImV4cCI6MTc3NTA1NTM0NCwiaWF0IjoxNzc1MDUxNzQ0LCJqdGkiOiJjNmE0Nzk2YS1iMzUwLTQzODQtODU3NS01NTljY2FiNDkwNWEiLCJ1c2VybmFtZSI6InRlc3R1c2VyIn0.W6uEHGktJT7WYMDswWFY4oBd9wsT5gMV7RiIn2t23RdX8nH1k1oaxvBLvX3kpRvU23HRM_KSqEYR

#### Create IAM Role for the Agents

In [12]:
from helpers.utils import create_agentcore_runtime_execution_role, AWS_CLAIMSVALIDATION_ROLE_NAME

execution_role_arn = create_agentcore_runtime_execution_role(AWS_CLAIMSVALIDATION_ROLE_NAME)

ℹ️ Role AWSClaimsValidationAssistantBedrockAgentCoreRole-us-east-1 already exists
Role ARN: arn:aws:iam::161615149547:role/AWSClaimsValidationAssistantBedrockAgentCoreRole-us-east-1


##### Configure and deploy our  agent:

In [13]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime_claimvalidation_agent = Runtime()
claimvalidation_agent_name="aws_legal_triage"

region = boto_session.region_name

# Configure the deployment
response_claimvalidation_agent = agentcore_runtime_claimvalidation_agent.configure(
    entrypoint="agents/triage-agent.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=claimvalidation_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="A2A",
)

print("Configuration completed:", response_claimvalidation_agent)




Entrypoint parsed: file=/home/ec2-user/SageMaker/multi-agent-arc-example/agents/triage-agent.py, bedrock_agentcore_name=triage-agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: aws_legal_triage
Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Generated Dockerfile: /home/ec2-user/SageMaker/multi-agent-arc-example/Dockerfile

Generated .dockerignore: /home/ec2-user/SageMaker/multi-agent-arc-example/.dockerignore
Keeping 'aws_legal_triage' as default agent
Bedrock AgentCore configured: /home/ec2-user/SageMaker/multi-agent-arc-example/.bedrock_agentcore.yaml


Configuration completed: config_path=PosixPath('/home/ec2-user/SageMaker/multi-agent-arc-example/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/home/ec2-user/SageMaker/multi-agent-arc-example/Dockerfile') dockerignore_path=PosixPath('/home/ec2-user/SageMaker/multi-agent-arc-example/.dockerignore') runtime='Docker' runtime_type=None region='us-east-1' account_id='161615149547' execution_role='arn:aws:iam::161615149547:role/AWSClaimsValidationAssistantBedrockAgentCoreRole-us-east-1' ecr_repository=None auto_create_ecr=True s3_path=None auto_create_s3=False memory_id=None network_mode='PUBLIC' network_subnets=None network_security_groups=None network_vpc_id=None


In [14]:
launch_result_claimvalidation = agentcore_runtime_claimvalidation_agent.launch()
print("Launch completed:", launch_result_claimvalidation.agent_arn)

claimvalidation_agent_arn = launch_result_claimvalidation.agent_arn

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'aws_legal_triage' to account 161615149547 (us-east-1)
Generated image tag: 20260401-135545-532
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: aws_legal_triage
ECR repository available: 161615149547.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-aws_legal_triage
Using execution role from config: arn:aws:iam::161615149547:role/AWSClaimsValidationAssistantBedrockAgentCoreRole-us-east-1
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 161615149547.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-aws_legal_triage


Getting or creating CodeBuild execution role for agent: aws_legal_triage
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-b361a9f5ee
Reusing existing CodeBuild execution role: arn:aws:iam::161615149547:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-b361a9f5ee
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: aws_legal_triage/source.zip
Updated CodeBuild project: bedrock-agentcore-aws_legal_triage-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.0s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.3s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE completed in 1.0s
🔄 BUILD started (total: 10s)
✅ BUILD completed in 40.3s
🔄 POST_BUILD started (total: 51s)
✅ POST_BUILD completed in 15.5s
🔄 COMPLETED started (total: 66s)
✅ COMPLETED completed in 1.0s
🎉 CodeBuild completed successfully in 1m 7s
CodeBuild completed succes

Launch completed: arn:aws:bedrock-agentcore:us-east-1:161615149547:runtime/aws_legal_triage-l7B8Z65tS5


In [15]:
status_response = agentcore_runtime_claimvalidation_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

Retrieved Bedrock AgentCore status for: aws_legal_triage


Final status: READY


#### Export and save outputs

Export variables to be used in next notebooks:

In [16]:
CLAIMSVALIDATION_AGENT_ID = launch_result_claimvalidation.agent_id
CLAIMSVALIDATION_AGENT_ARN = launch_result_claimvalidation.agent_arn
CLAIMSVALIDATION_AGENT_NAME = claimvalidation_agent_name


CLAIMSVALIDATION_COGNITO_CLIENT_ID = cognito_config.get("client_id")
CLAIMSVALIDATION_COGNITO_SECRET = cognito_config.get("client_secret")
CLAIMSVALIDATION_DISCOVERY_URL = cognito_config.get("discovery_url")

%store CLAIMSVALIDATION_AGENT_ID
%store CLAIMSVALIDATION_AGENT_ARN
%store CLAIMSVALIDATION_AGENT_NAME
%store CLAIMSVALIDATION_COGNITO_CLIENT_ID
%store CLAIMSVALIDATION_COGNITO_SECRET
%store CLAIMSVALIDATION_DISCOVERY_URL

Stored 'CLAIMSVALIDATION_AGENT_ID' (str)
Stored 'CLAIMSVALIDATION_AGENT_ARN' (str)
Stored 'CLAIMSVALIDATION_AGENT_NAME' (str)
Stored 'CLAIMSVALIDATION_COGNITO_CLIENT_ID' (str)
Stored 'CLAIMSVALIDATION_COGNITO_SECRET' (str)
Stored 'CLAIMSVALIDATION_DISCOVERY_URL' (str)


In [17]:
from helpers.utils import put_ssm_parameter, SSM_CLAIMSVALIDATION_AGENT_ARN

put_ssm_parameter(SSM_CLAIMSVALIDATION_AGENT_ARN, claimvalidation_agent_arn)

### Invoking A2A agents

Firstly, let's refresh the auth token:

In [18]:
bearer_token = reauthenticate_user(
    cognito_config.get("client_id"), 
    cognito_config.get("client_secret")
)

In [19]:
cognito_config.get("client_id")

'1m2dej9j10v39aml3tcabdpl3t'

In [20]:
CLAIMSVALIDATION_COGNITO_CLIENT_ID

'1m2dej9j10v39aml3tcabdpl3t'

In [21]:
from uuid import uuid4
from urllib.parse import quote

def fetch_agent_card(agent_arn):
    # URL encode the agent ARN
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/.well-known/agent-card.json"
    
    # Generate a unique session ID
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # Set headers
    headers = {
        'Accept': '*/*',
        'Authorization': f'Bearer {bearer_token}',
        'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
    }

    try:
        # Make the request
        response = requests.get(url, headers=headers, timeout=900)
        response.raise_for_status()

        # Parse and pretty print JSON
        agent_card = response.json()
        print(json.dumps(agent_card, indent=2))

        return agent_card

    except requests.exceptions.RequestException as e:
        print(f"Error fetching agent card: {e}")
        raise e

In [22]:
fetch_agent_card(claimvalidation_agent_arn)

Generated session ID: f7e08754-e1f9-4507-afe2-99350871d4ba
{
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "A Single Agent with ARC Legal Triage Validation",
  "name": "Legal Triage - ARC - NO RAG",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [],
  "url": "https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A161615149547%3Aruntime%2Faws_legal_triage-l7B8Z65tS5/invocations/",
  "version": "0.0.1"
}


{'capabilities': {'streaming': True},
 'defaultInputModes': ['text'],
 'defaultOutputModes': ['text'],
 'description': 'A Single Agent with ARC Legal Triage Validation',
 'name': 'Legal Triage - ARC - NO RAG',
 'preferredTransport': 'JSONRPC',
 'protocolVersion': '0.3.0',
 'skills': [],
 'url': 'https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A161615149547%3Aruntime%2Faws_legal_triage-l7B8Z65tS5/invocations/',
 'version': '0.0.1'}

#### Test agents

Now, let's invoke the first agent, using A2A:

In [23]:
import asyncio
import logging
import os
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 300  # set request timeout to 5 minutes

def format_agent_response(response):
    """Extract and format agent response for human readability."""
    # Get the main response text from artifacts
    if response.artifacts and len(response.artifacts) > 0:
        artifact = response.artifacts[0]
        if artifact.parts and len(artifact.parts) > 0:
            return artifact.parts[0].root.text
    
    # Fallback: concatenate all agent messages from history
    agent_messages = [
        msg.parts[0].root.text 
        for msg in response.history 
        if msg.role.value == 'agent' and msg.parts
    ]
    return ''.join(agent_messages)


def create_message(*, role: Role = Role.user, text: str) -> Message:
    return Message(
        kind="message",
        role=role,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_sync_message(agent_arn, message: str):
    # Get runtime URL from environment variable
    escaped_agent_arn = quote(agent_arn, safe='')

    # Construct the URL
    runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations/"
    
    # Generate a unique session ID
    session_id = str(uuid4())
    print(f"Generated session ID: {session_id}")

    # Add authentication headers for AgentCore
    headers = {"Authorization": f"Bearer {bearer_token}",
              'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id}
        
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, headers=headers) as httpx_client:
        # Get agent card from the runtime URL
        resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
        agent_card = await resolver.get_agent_card()
        print(agent_card)

        # Agent card contains the correct URL (same as runtime_url in this case)
        # No manual override needed - this is the path-based mounting pattern

        # Create client using factory
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=False,  # Use non-streaming mode for sync response
        )
        factory = ClientFactory(config)
        client = factory.create(agent_card)

        # Create and send message
        msg = create_message(text=message)

        print(msg)

        # With streaming=False, this will yield exactly one result
        async for event in client.send_message(msg):
            if isinstance(event, Message):
                logger.info(event.model_dump_json(exclude_none=True, indent=2))
                return event
            elif isinstance(event, tuple) and len(event) == 2:
                # (Task, UpdateEvent) tuple
                task, update_event = event
                logger.info(f"Task: {task.model_dump_json(exclude_none=True, indent=2)}")
                if update_event:
                    logger.info(f"Update: {update_event.model_dump_json(exclude_none=True, indent=2)}")
                return task
            else:
                # Fallback for other response types
                logger.info(f"Response: {str(event)}")
                return event

In [24]:
# Create user prompt with vehicle and damage information
# validation_input_prompt = f"""
# Validate this Claim request. Carefully read and use the {json.dumps(fnol_data, indent=2)}  provided to extract the required information to validate the claim
# Carefully review each field to extract the required information to validate the claim. See the following <field_examples>:

# <field_examples>
# **fnol:** 
# - reportDate
# - reportTime": "14:35:00",
# - reportMethod": "Online",
# - submittedBy": "Policyholder"
    
# **Claim Information:**
# - Claim Number
# - Incident Date
# - Incident Description

# **Insured Vehicle Information:**
# - Year
# - Make
# - Model
# - VIN
# - Mileage: miles

# **Damage Assessment:**
# - Damaged Areas
# - Damage Description
# - Vehicle Drivable
# </field_examples>

# Ouput: include a section that shows
# ## **CLAIM VALIDATION RESULT: INVALID vs VALID**
# If RESULT is INVALID, provide explanation on why it was invalid and list the rules that were not met
# """

print("=" * 80)
print("USER PROMPT CREATED")
print("=" * 80)
# print(appraisal_input_prompt)
print("=" * 80)
print("\n📝 Prompt ready for agent execution")
print("\n" + "=" * 80)

USER PROMPT CREATED

📝 Prompt ready for agent execution



In [25]:
triage_report = f""" 
Validate this <Triage Report>. Carefully read and use each field provided to extract the required information to validate the Triage Report


<Triage Report>
Adjuster: Michael Torres 
Manager: Sandra Whitfield USRM - NY
People/Entities Involved
* Linda Hargrove – First Named Insured
* Robert Hargrove – Spouse of Named Insured
* Teresa Campos – Claimant (sustained injuries as a guest on the insured premises)
* Brightway Pool Services, LLC – Third-party contractor performing maintenance at time of loss
Coverage Notes and Facts of Loss
DOL: 03/14/2026
Loss Location: 482 Cedarbrook Lane, Westchester, NY 10501
On March 14, 2026, Teresa Campos was visiting the Hargrove residence for a social gathering. Brightway Pool Services, LLC was on-site performing scheduled pool maintenance at the time of the event. Ms. Campos slipped on a wet surface surrounding the pool area that had been recently pressure-washed by the contractor and sustained a fractured left wrist and contusions to her left hip. Emergency medical services were called and Ms. Campos was transported to Northern Westchester Hospital, where she was treated and released the same day. Ms. Campos has since retained counsel (Berman & Kline, LLP) and submitted a demand for medical expenses, lost wages, and pain and suffering totaling $185,000. The Named Insured reported the loss on 03/16/2026. Brightway Pool Services, LLC maintains its own commercial general liability policy through Beacon Mutual Insurance (Policy No. BM-CGL-2024-09821). The Hargrove residence is covered under a homeowners policy (Policy No. HO-7741928) with an effective period of 01/01/2026 – 01/01/2027 and a personal umbrella policy (Policy No. PUP-553210) with an effective period of 01/01/2025 – 01/01/2026. Both policies are underwritten by our company.
Adjuster Question: What is the priority of coverage between the Hargrove Homeowners Policy (HO-7741928), the Hargrove Personal Umbrella Policy (PUP-553210), and the Brightway Pool Services CGL Policy (BM-CGL-2024-09821)? Specifically, does the contractor's CGL policy respond as primary, and do the Hargrove policies apply on an excess basis, or is there a concurrent obligation?
Documents Relevant to Coverage Question
* Homeowners Policy HO-7741928 (full policy jacket including declarations, insuring agreement, exclusions, and endorsements)
* Personal Umbrella Policy PUP-553210 (full policy jacket including declarations, insuring agreement, drop-down provisions, and endorsements)
* Brightway Pool Services CGL Policy BM-CGL-2024-09821 (copy of declarations page and additional insured endorsement, if any)
* Service Agreement between Linda Hargrove and Brightway Pool Services, LLC dated 02/01/2026 (including indemnification and insurance requirements provisions)
* Incident/Loss Report filed by Named Insured on 03/16/2026
* Northern Westchester Hospital Emergency Room records for Teresa Campos dated 03/14/2026
* Demand letter from Berman & Kline, LLP dated 04/02/2025
* Photographs of the pool area and surrounding surface taken on 03/15/2026
Missing Materials Requested from Adjuster: None
</Triage Report>
"""
print("=" * 80)
print("USER PROMPT CREATED")
print("=" * 80)
# print(appraisal_input_prompt)
print("=" * 80)
print("\n📝 Prompt ready for agent execution")
print("\n" + "=" * 80)

USER PROMPT CREATED

📝 Prompt ready for agent execution



In [28]:
result = await send_sync_message(claimvalidation_agent_arn,triage_report )
formatted_output = format_agent_response(result)
print(formatted_output)

Generated session ID: 3b1a143d-a07a-4ea5-a1c3-17e1644be622


A2AClientHTTPError: HTTP Error 401: Failed to fetch agent card from https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A161615149547%3Aruntime%2Faws_legal_triage-l7B8Z65tS5/invocations/.well-known/agent-card.json: Client error '401 Unauthorized' for url 'https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A161615149547%3Aruntime%2Faws_legal_triage-l7B8Z65tS5/invocations/.well-known/agent-card.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [ ]:
# result = await send_sync_message(claimvalidation_agent_arn, "I was delivering goods for my company using my personal car. I got into an accident, this was over 3 months ago. Does my policy cover this?")
# formatted_output = format_agent_response(result)
# print(formatted_output)